# Pagila Consumer Market Insights Analytics

Connect to the Pagila database and complete analysis.

In [1]:
# import all required libraries
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, URL

In [2]:
# establish database connection
class DBConn:
    """ Establish database connection. """
    def __init__(self, drivername, database, host, port, username, password):
        """ Database connection parameters. """
        self.url = URL.create(
            drivername = drivername,
            database = database,
            host = host,
            port = port,
            username = username,
            password = password,
        )
        self.engine = create_engine(self.url)

    def execute_sql(self, sql):
        """ Excute SQL and store in DataFrame. """
        return pd.read_sql(sql, self.engine)

    def close_connection(self):
        """ Close database connection. """
        self.engine.dispose()

In [3]:
# retrieve data from the database
pagila = DBConn(
    drivername = "postgresql+psycopg2",
    database = "pagila",
    host = "localhost",
    port = "5432",
    username = "postgres",
    password = "LZZcode2026!"
)

# SQL queries to extract data
rental = pagila.execute_sql(
    """
    SELECT
        customer_id, 
        rental_date 
    FROM rental
    """
)

customer = pagila.execute_sql(
    """
    SELECT
        c.customer_id,
        c.store_id,
        p.amount
    FROM customer AS c
    JOIN payment AS p
        ON c.customer_id = p.customer_id
    """
)

payment = pagila.execute_sql(
    """
    SELECT
        customer_id,
        payment_date,
        amount
    FROM payment
    """
)

ex3_customer = pagila.execute_sql(
    """
	SELECT
		customer_id,
		store_id
	FROM customer
    """
)

ex3_payment = pagila.execute_sql(
    """
	SELECT
		customer_id,
		amount
	FROM payment
    """
)

ex4_rental = pagila.execute_sql(
	"""
SELECT customer_id, rental_date	
FROM rental
	"""
)


# dispose engine and close connection
pagila.close_connection()

In [50]:
rental.head()

,customer_id,rental_date
0,459,2022-05-24 21:54:33+00:00
1,408,2022-05-24 22:03:39+00:00
2,333,2022-05-24 22:04:41+00:00
3,222,2022-05-24 22:05:21+00:00
4,549,2022-05-24 22:08:07+00:00


In [51]:
rental.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16044 entries, 0 to 16043
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   customer_id  16044 non-null  int64              
 1   rental_date  16044 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), int64(1)
memory usage: 250.8 KB


In [61]:
# convert to month
rental['month'] = rental['rental_date'].dt.to_period('M')

# get last active month
last_active_month = rental.groupby('customer_id')['month'].max().reset_index()

# get the latest month
latest_month = rental['month'].max()

# classify customer status
last_active_month['status'] = last_active_month['month'].apply(
    lambda x: 'Churned' if latest_month > x else 'Active' 
)

last_active_month

C:\Users\lizha\AppData\Local\Temp\ipykernel_21976\3042329345.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  rental['month'] = rental['rental_date'].dt.to_period('M')


,customer_id,month,status
0,1,2022-08,Active
1,2,2022-08,Active
2,3,2022-08,Active
3,4,2022-08,Active
4,5,2022-08,Active
...,...,...,...
594,595,2022-08,Active
595,596,2022-08,Active
596,597,2022-08,Active
597,598,2022-08,Active


In [63]:
customer.head()

,customer_id,store_id,amount
0,269,1,0.99
1,274,1,2.99
2,297,1,0.99
3,344,1,2.99
4,348,2,0.99


In [64]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16049 entries, 0 to 16048
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  16049 non-null  int64  
 1   store_id     16049 non-null  int64  
 2   amount       16049 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 376.3 KB


In [116]:
# calculate store totals
store_revenue = customer.groupby('store_id').agg(
    store_total=('amount', 'sum')
).reset_index()

# calculate customer spending at each store
customer_store_spending = customer.groupby(['customer_id', 'store_id']).agg(
    total_spent=('amount', 'sum')
).reset_index()

# merge the two data sets
merge_data = store_revenue.merge(customer_store_spending, on='store_id')

# calculate the percentage of customer contribution to store total
merge_data['pct_contribution'] = merge_data['total_spent'] / merge_data['store_total']

# rank customer spending in stores.
merge_data['rank_in_store'] = merge_data.groupby(['store_id'])['total_spent'].rank(
    method='dense', ascending=False
)

# segment the customers
merge_data['segment'] = merge_data['pct_contribution'].apply(
    lambda x : 'Dominant' if x >= 0.3 else
    ('Significant' if x >= 0.15 else 'Normal')
)

merge_data

,store_id,store_total,customer_id,total_spent,pct_contribution,rank_in_store,segment
0,1,37001.52,1,118.68,0.003207,109.0,Normal
1,1,37001.52,2,128.73,0.003479,75.0,Normal
2,1,37001.52,3,135.74,0.003668,52.0,Normal
3,1,37001.52,5,144.62,0.003908,32.0,Normal
4,1,37001.52,7,151.67,0.004099,20.0,Normal
...,...,...,...,...,...,...,...
594,2,30414.99,582,113.75,0.003740,104.0,Normal
595,2,30414.99,584,129.70,0.004264,58.0,Normal
596,2,30414.99,590,112.75,0.003707,109.0,Normal
597,2,30414.99,593,113.74,0.003740,105.0,Normal


In [93]:
payment.head()

,customer_id,amount,payment_date
0,269,0.99,2022-01-29 01:58:52.222594+00:00
1,274,2.99,2022-01-25 12:14:16.895377+00:00
2,297,0.99,2022-01-28 00:49:49.128218+00:00
3,344,2.99,2022-01-31 05:58:51.176578+00:00
4,348,0.99,2022-01-26 16:52:41.359433+00:00


In [94]:
payment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16049 entries, 0 to 16048
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   customer_id   16049 non-null  int64              
 1   amount        16049 non-null  float64            
 2   payment_date  16049 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(1), int64(1)
memory usage: 376.3 KB


In [112]:
# convert to month
payment['month'] = payment['payment_date'].dt.to_period('M')

# get customer historical average
avg_amount = payment.groupby(['customer_id', 'month'], as_index=False).agg(
    avg_amount = ('amount', 'mean')
)

# get merge data sets
merge_amount = payment.merge(avg_amount, on='customer_id', how='inner')

# classify customers
merge_amount['risk_flag'] = 'Low Risk'
merge_amount.loc[merge_amount['amount'] >= 3 * merge_amount['avg_amount'], 'risk_flag'] = 'High Risk'
merge_amount.loc[merge_amount['amount'] >= 2 * merge_amount['avg_amount'], 'risk_flag'] = 'Medium Risk'

merge_amount

C:\Users\lizha\AppData\Local\Temp\ipykernel_21976\607917309.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  payment['month'] = payment['payment_date'].dt.to_period('M')


,customer_id,payment_id,amount,payment_date,month_x,month_y,avg_amount,risk_flag
0,269,16051,0.99,2022-01-29 01:58:52.222594+00:00,2022-01,2022-01,1.990000,Low Risk
1,269,16051,0.99,2022-01-29 01:58:52.222594+00:00,2022-01,2022-02,4.490000,Low Risk
2,269,16051,0.99,2022-01-29 01:58:52.222594+00:00,2022-01,2022-03,2.990000,Low Risk
3,269,16051,0.99,2022-01-29 01:58:52.222594+00:00,2022-01,2022-04,4.156667,Low Risk
4,269,16051,0.99,2022-01-29 01:58:52.222594+00:00,2022-01,2022-05,4.592000,Low Risk
...,...,...,...,...,...,...,...,...
106150,264,32098,2.99,2022-07-06 22:14:23.213321+00:00,2022-07,2022-03,2.990000,Low Risk
106151,264,32098,2.99,2022-07-06 22:14:23.213321+00:00,2022-07,2022-04,5.132857,Low Risk
106152,264,32098,2.99,2022-07-06 22:14:23.213321+00:00,2022-07,2022-05,2.190000,Low Risk
106153,264,32098,2.99,2022-07-06 22:14:23.213321+00:00,2022-07,2022-06,5.656667,Low Risk


In [115]:
merge_amount['risk_flag'].unique()

array(['Low Risk', 'Medium Risk'], dtype=object)

In [118]:
ex1_pmt.head()

,customer_id,payment_date,amount
0,269,2022-01-29 01:58:52.222594+00:00,0.99
1,274,2022-01-25 12:14:16.895377+00:00,2.99
2,297,2022-01-28 00:49:49.128218+00:00,0.99
3,344,2022-01-31 05:58:51.176578+00:00,2.99
4,348,2022-01-26 16:52:41.359433+00:00,0.99


In [119]:
ex1_pmt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16049 entries, 0 to 16048
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   customer_id   16049 non-null  int64              
 1   payment_date  16049 non-null  datetime64[ns, UTC]
 2   amount        16049 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(1), int64(1)
memory usage: 376.3 KB


In [123]:
# compute monthly spending
ex1_pmt['month'] = ex1_pmt['payment_date'].dt.to_period('M')
monthly_spent = ex1_pmt.groupby(['customer_id','month'], as_index=False).agg(
    monthly_spent = ('amount', 'sum')
)

# compute average monthly spent
avg_spent = monthly_spent.groupby(['customer_id'], as_index=False).agg(
    avg_spent = ('monthly_spent', 'mean')
)

# merge the two datasets and compare
ex1_merge = monthly_spent.merge(avg_spent, on='customer_id', how='inner')

# create risk_flags and classify
ex1_merge['risk_flag'] = 'Low Risk'
ex1_merge.loc[ex1_merge['monthly_spent'] > 2 * ex1_merge['avg_spent'], 'risk_flag'] = 'High Risk'
ex1_merge.loc[ex1_merge['monthly_spent'] > 1.5 * ex1_merge['avg_spent'], 'risk_flag'] = 'Medium Risk'

# display the resultant table
ex1_merge.head()

C:\Users\lizha\AppData\Local\Temp\ipykernel_21976\946759759.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ex1_pmt['month'] = ex1_pmt['payment_date'].dt.to_period('M')


,customer_id,month,monthly_spent,avg_spent,risk_flag
0,1,2022-01,9.98,16.954286,Low Risk
1,1,2022-02,17.96,16.954286,Low Risk
2,1,2022-03,10.97,16.954286,Low Risk
3,1,2022-04,32.93,16.954286,Medium Risk
4,1,2022-05,14.96,16.954286,Low Risk


In [124]:
ex1_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3946 entries, 0 to 3945
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype    
---  ------         --------------  -----    
 0   customer_id    3946 non-null   int64    
 1   month          3946 non-null   period[M]
 2   monthly_spent  3946 non-null   float64  
 3   avg_spent      3946 non-null   float64  
 4   risk_flag      3946 non-null   object   
dtypes: float64(2), int64(1), object(1), period[M](1)
memory usage: 154.3+ KB


In [125]:
ex1_pmt["month"] = pd.to_datetime(ex1_pmt["payment_date"]).dt.to_period("M")

monthly = ex1_pmt.groupby(["customer_id", "month"], as_index=False)["amount"].sum()
monthly.rename(columns={"amount": "monthly_spent"}, inplace=True)

monthly["avg_spent"] = monthly.groupby("customer_id")["monthly_spent"].transform("mean")

monthly["risk_flag"] = "Low Risk"
monthly.loc[monthly["monthly_spent"] >= 1.5 * monthly["avg_spent"], "risk_flag"] = "Medium Risk"
monthly.loc[monthly["monthly_spent"] >= 2 * monthly["avg_spent"], "risk_flag"] = "High Risk"

C:\Users\lizha\AppData\Local\Temp\ipykernel_21976\2633332328.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ex1_pmt["month"] = pd.to_datetime(ex1_pmt["payment_date"]).dt.to_period("M")


In [126]:
monthly.head()

,customer_id,month,monthly_spent,avg_spent,risk_flag
0,1,2022-01,9.98,16.954286,Low Risk
1,1,2022-02,17.96,16.954286,Low Risk
2,1,2022-03,10.97,16.954286,Low Risk
3,1,2022-04,32.93,16.954286,Medium Risk
4,1,2022-05,14.96,16.954286,Low Risk


In [127]:
monthly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3946 entries, 0 to 3945
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype    
---  ------         --------------  -----    
 0   customer_id    3946 non-null   int64    
 1   month          3946 non-null   period[M]
 2   monthly_spent  3946 non-null   float64  
 3   avg_spent      3946 non-null   float64  
 4   risk_flag      3946 non-null   object   
dtypes: float64(2), int64(1), object(1), period[M](1)
memory usage: 154.3+ KB


In [130]:
payment.head()

,customer_id,payment_date,amount
0,269,2022-01-29 01:58:52.222594+00:00,0.99
1,274,2022-01-25 12:14:16.895377+00:00,2.99
2,297,2022-01-28 00:49:49.128218+00:00,0.99
3,344,2022-01-31 05:58:51.176578+00:00,2.99
4,348,2022-01-26 16:52:41.359433+00:00,0.99


In [132]:
# compute monthly spending
payment['month'] = payment['payment_date'].dt.to_period('M')
monthly_spent = payment.groupby(['customer_id', 'month'], as_index=False).agg(
    monthly_spent = ('amount', 'sum')
)
# compute previous month spending
monthly_spent = monthly_spent.sort_values(['customer_id', 'month'])
monthly_spent['prev_spent'] = monthly_spent.groupby(['customer_id'])['monthly_spent'].shift(1) # learn this
# add risk flags
monthly_spent['risk_flag'] = 'Low Risk'
monthly_spent.loc[monthly_spent['monthly_spent'] >= 2 * monthly_spent['prev_spent'], 'risk_flag'] = 'High Risk'
monthly_spent.loc[monthly_spent['monthly_spent'] >= 1.5 * monthly_spent['prev_spent'], 'risk_flag'] = 'Medium Risk'
# display results
monthly_spent.head()

C:\Users\lizha\AppData\Local\Temp\ipykernel_21976\246180707.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  payment['month'] = payment['payment_date'].dt.to_period('M')


,customer_id,month,monthly_spent,prev_spent,risk_flag
0,1,2022-01,9.98,NaN,Low Risk
1,1,2022-02,17.96,9.98,Medium Risk
2,1,2022-03,10.97,17.96,Low Risk
3,1,2022-04,32.93,10.97,Medium Risk
4,1,2022-05,14.96,32.93,Low Risk


In [148]:
# merge the two data sets
ex3_merge = ex3_customer.merge(ex3_payment, on="customer_id", how="inner")
# calculate store total
store_revenue = ex3_merge.groupby("store_id").agg(
	store_total = ("amount", "sum")
).reset_index()
# calculate customer spending in each store
customer_spending = ex3_merge.groupby(["customer_id", "store_id"], as_index=False).agg(
	total_spend = ("amount", "sum")
)
# merge store_revenue and customer_spending
ex3_new = store_revenue.merge(customer_spending, on="store_id", how="inner")
ex3_new = ex3_new.sort_values(by=['store_id','total_spend'])
# calculate the pct_contribution
ex3_new["pct_contribution"] = ex3_new["total_spend"] / ex3_new["store_total"]
# rank order pct_contribution
ex3_new["rank"] = ex3_new.groupby("store_id")["pct_contribution"].rank(method = "dense", ascending=False)
ex3_new.head()

,store_id,store_total,customer_id,total_spend,pct_contribution,rank
139,1,37001.52,248,50.85,0.001374,272.0
173,1,37001.52,318,52.88,0.001429,271.0
315,1,37001.52,586,64.81,0.001752,270.0
251,1,37001.52,458,66.81,0.001806,269.0
121,1,37001.52,218,67.82,0.001833,268.0


In [143]:
ex3_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 599 entries, 562 to 139
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   store_id          599 non-null    int64  
 1   store_total       599 non-null    float64
 2   customer_id       599 non-null    int64  
 3   total_spend       599 non-null    float64
 4   pct_contribution  599 non-null    float64
 5   rank              599 non-null    float64
dtypes: float64(4), int64(2)
memory usage: 32.8 KB


In [12]:
# latest month
ex4_rental["month"] = ex4_rental["rental_date"].dt.to_period("M")
latest_month = ex4_rental["month"].max()
# latest active month
active_month = ex4_rental.groupby("customer_id")["month"].max().rename(columns=({"month" : "last_active_month"}))
# merge and compare
ex4_merge = ex4_rental.merge(active_month, on="customer_id", how="inner")
ex4_merge["latest_month"] = latest_month
# compare and assign status
ex4_merge["status"] = ex4_merge["latest_month"].apply(lambda x : "Active" if x == ex4_merge["last_active_month"] else "Inactive")


C:\Users\lizha\AppData\Local\Temp\ipykernel_39772\3951253153.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  ex4_rental["month"] = ex4_rental["rental_date"].dt.to_period("M")


TypeError: Series.rename() got an unexpected keyword argument 'columns'

In [10]:
active_month

customer_id
1      2022-08
2      2022-08
3      2022-08
4      2022-08
5      2022-08
        ...   
595    2022-08
596    2022-08
597    2022-08
598    2022-08
599    2022-08
Name: month, Length: 599, dtype: period[M]